# 🎙️ Ses İşareti Analizi ve Cinsiyet Sınıflandırma
## Audio Signal Analysis & Gender Classification
### 2025-2026 Bahar Dönemi — Dönemiçi Proje

**Yöntem:** Otokorelasyon tabanlı F0 tespiti + Kural Tabanlı Sınıflandırıcı  
**Kütüphaneler:** NumPy, SciPy, Pandas, Matplotlib, Seaborn

In [ ]:
# ===========================================================
# HÜCRE 1: Kütüphane İmportları ve Yol Ayarları
# ===========================================================
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scipy.io.wavfile as wav
import scipy.signal as signal
import warnings
warnings.filterwarnings('ignore')

# Proje kök dizinini Python path'e ekle
project_root = os.path.abspath('..')
sys.path.insert(0, project_root)

from src.feature_extraction import (
    load_audio, frame_signal,
    compute_short_time_energy, compute_zcr,
    detect_voiced_frames,
    estimate_f0_autocorrelation, estimate_f0_fft,
    compute_autocorrelation_fast, compute_autocorrelation,
    extract_features
)
from src.classifier  import classify_gender, compute_metrics, THRESHOLDS
from src.dataset_loader import load_all_metadata, print_dataset_summary
from src.visualizer  import (
    plot_full_analysis, plot_autocorrelation_vs_fft,
    plot_f0_distributions, plot_confusion_matrix,
    plot_results_table
)

plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print('✅ Tüm modüller başarıyla yüklendi!')

---
## BÖLÜM 1: Veri Seti Yükleme

In [ ]:
# ===========================================================
# HÜCRE 2: Veri Seti ve Metadata Yükleme
# ===========================================================
DATASET_PATH = '../Dataset'  # Dataset klasörünüzün yolu
SR = 16000                   # Örnekleme hızı (Hz)

# Tüm grup Excel dosyalarını okuyup birleştir
df_meta = load_all_metadata(DATASET_PATH)
print_dataset_summary(df_meta)

# Metadata tablosunu göster
print('\nİlk 5 kayıt:')
df_meta.head()

In [ ]:
# ===========================================================
# HÜCRE 3: Veri Seti Demografik Grafikleri
# ===========================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Dataset Demographics (Veri Seti Demografisi)', fontsize=13, fontweight='bold')

# Cinsiyet dağılımı
colors = {'Erkek': '#2196F3', 'Kadın': '#E91E63', 'Çocuk': '#4CAF50'}
if 'Cinsiyet' in df_meta.columns:
    gender_counts = df_meta['Cinsiyet'].value_counts()
    gender_counts.plot(kind='bar', ax=axes[0],
                       color=[colors.get(c, '#888') for c in gender_counts.index],
                       edgecolor='white', linewidth=1.5)
    axes[0].set_title('Gender Distribution (Cinsiyet Dağılımı)')
    axes[0].set_xlabel('Gender')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    for bar in axes[0].patches:
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     str(int(bar.get_height())), ha='center', fontweight='bold')

# Grup dağılımı
if 'Grup' in df_meta.columns:
    group_counts = df_meta['Grup'].value_counts()
    group_counts.plot(kind='bar', ax=axes[1], color='#78909C',
                      edgecolor='white', linewidth=1.5)
    axes[1].set_title('Group Distribution (Grup Dağılımı)')
    axes[1].set_xlabel('Group')
    axes[1].set_ylabel('Count')
    axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## BÖLÜM 2: Zaman Düzlemi Analizi — STE, ZCR, Sesli/Sessiz Ayrımı

In [ ]:
# ===========================================================
# HÜCRE 4: Tek Dosya Zaman Düzlemi Analizi
# ===========================================================
# İlk geçerli ses dosyasını yükle
sample_path = None
for _, row in df_meta.iterrows():
    p = str(row.get('Dosya_Path', ''))
    if os.path.exists(p):
        sample_path = p
        sample_gender = str(row.get('Cinsiyet', '?'))
        break

if sample_path is None:
    print('⚠️ Ses dosyası bulunamadı!')
else:
    print(f'📁 Analiz edilen dosya: {os.path.basename(sample_path)}')
    print(f'👤 Cinsiyet (Ground Truth): {sample_gender}')
    
    # Ses yükleme
    audio, sr_loaded = load_audio(sample_path, target_sr=SR)
    print(f'⏱️  Süre: {len(audio)/sr_loaded:.2f} saniye')
    print(f'🎚️  Örnekleme hızı: {sr_loaded} Hz')
    print(f'📊  Örnek sayısı: {len(audio)}')
    
    # Pencereleme (25 ms çerçeve, 10 ms atlama)
    frames, frame_len, hop_len = frame_signal(audio, sr_loaded, frame_ms=25, hop_ms=10)
    print(f'\n🪟 Çerçeve boyutu: {frame_len} örnek ({frame_len/sr_loaded*1000:.0f} ms)')
    print(f'↔️  Atlama adımı: {hop_len} örnek ({hop_len/sr_loaded*1000:.0f} ms)')
    print(f'📦  Toplam çerçeve sayısı: {len(frames)}')

In [ ]:
# ===========================================================
# HÜCRE 5: Enerji, ZCR ve Sesli Bölge Analizi
# ===========================================================
if sample_path:
    # Hesaplamalar
    energy = compute_short_time_energy(frames)
    zcr    = compute_zcr(frames, sr_loaded)
    voiced = detect_voiced_frames(energy, zcr)
    
    print(f'🔊 Sesli çerçeve sayısı: {voiced.sum()} / {len(voiced)}')
    print(f'📈 Sesli çerçeve oranı: {voiced.mean()*100:.1f}%')
    print(f'⚡ Ortalama enerji (sesli): {energy[voiced].mean():.4e}')
    print(f'〰️  Ortalama ZCR (sesli): {zcr[voiced].mean():.0f} Hz')
    
    # Tam analiz grafiği
    fig = plot_full_analysis(
        audio=audio, sr=sr_loaded,
        energy=energy, zcr=zcr,
        voiced_mask=voiced,
        frames=frames, hop_length=hop_len,
        title=f'Full Signal Analysis — {sample_gender} Voice'
    )
    plt.show()

---
## BÖLÜM 3: Otokorelasyon ile F0 Tespiti

In [ ]:
# ===========================================================
# HÜCRE 6: Otokorelasyon Araştırması — R(τ) Hesabı
# ===========================================================
"""
Otokorelasyon Fonksiyonu:
  R[τ] = Σ x[n] · x[n-τ]
  
Periyodik bir sinyal için:
  - τ = 0 → maksimum R[0] (sinyalin kendisiyle korelasyonu)
  - τ = T (periyot) → ikinci maksimum
  - F0 = sr / τ_peak
"""

if sample_path and voiced.sum() > 0:
    # En sesli çerçeveyi seç
    voiced_indices  = np.where(voiced)[0]
    best_frame_idx  = voiced_indices[np.argmax(energy[voiced_indices])]
    sample_frame    = frames[best_frame_idx]
    
    print(f'🎯 Seçilen çerçeve: #{best_frame_idx} '
          f'(t = {best_frame_idx*hop_len/sr_loaded:.3f} s)')
    
    # Otokorelasyon hesabı (FFT tabanlı hızlı versiyon)
    acf = compute_autocorrelation_fast(sample_frame)
    
    # F0 tespiti — otokorelasyon yöntemi
    f0_autocorr = estimate_f0_autocorrelation(sample_frame, sr_loaded,
                                               f0_min=60, f0_max=500)
    # F0 tespiti — FFT yöntemi (karşılaştırma için)
    f0_fft = estimate_f0_fft(sample_frame, sr_loaded, f0_min=60, f0_max=500)
    
    print(f'\n🎵 F0 (Otokorelasyon): {f0_autocorr:.2f} Hz')
    print(f'🎵 F0 (FFT):           {f0_fft:.2f} Hz')
    print(f'📏 Fark:               {abs(f0_autocorr-f0_fft):.2f} Hz')
    if f0_autocorr > 0:
        print(f'⏱️  Periyot (T):        {1000/f0_autocorr:.2f} ms')

In [ ]:
# ===========================================================
# HÜCRE 7: Otokorelasyon vs FFT Görsel Karşılaştırma
# ===========================================================
if sample_path and voiced.sum() > 0:
    fig = plot_autocorrelation_vs_fft(
        frame=sample_frame,
        sr=sr_loaded,
        acf=acf,
        f0_autocorr=f0_autocorr,
        f0_fft=f0_fft,
        title=f'Autocorrelation vs FFT Comparison — {sample_gender} Voice'
    )
    plt.show()
    
    print('\n📝 YORUM / INTERPRETATION:')
    print('  Otokorelasyon fonksiyonunda τ = periyot değerinde belirgin tepe görülmektedir.')
    print('  Bu tepe noktası FFT spektrumundaki ilk harmonikle (F0) örtüşmektedir.')
    print(f'  İki yöntem arası fark {abs(f0_autocorr-f0_fft):.1f} Hz olup,'
          '  frekans çözünürlük sınırları dahilindedir.')

---
## BÖLÜM 4: Tüm Veri Seti Üzerinde Öznitelik Çıkarımı

In [ ]:
# ===========================================================
# HÜCRE 8: Toplu Öznitelik Çıkarımı (Tüm Veri Seti)
# ===========================================================
print('🔄 Tüm dosyalar işleniyor...')
results = []

for idx, (_, row) in enumerate(df_meta.iterrows()):
    filepath = str(row.get('Dosya_Path', ''))
    gender   = str(row.get('Cinsiyet', 'Unknown'))
    
    print(f'  [{idx+1}/{len(df_meta)}] {os.path.basename(filepath)}...', end=' ')
    
    if not os.path.exists(filepath):
        print('❌ Dosya bulunamadı')
        continue
    
    features = extract_features(filepath, sr=SR)
    
    if 'error' not in features:
        pred, conf, reason = classify_gender(features)
        features.update({
            'Dosya_Path' : filepath,
            'Cinsiyet'   : gender,
            'Grup'       : str(row.get('Grup', 'Unknown')),
            'prediction' : pred,
            'confidence' : conf,
            'reasoning'  : reason,
            'correct'    : (pred == gender)
        })
        results.append(features)
        print(f'F0={features["f0_mean"]:.0f}Hz, pred={pred} '
              f'{"✓" if pred==gender else "✗"}')
    else:
        print(f'⚠️ {features["error"]}')

df_results = pd.DataFrame(results)
print(f'\n✅ İşleme tamamlandı: {len(df_results)} dosya analiz edildi.')
df_results[['Cinsiyet','prediction','f0_mean','f0_std','zcr_mean','confidence','correct']].head(10)

---
## BÖLÜM 5: İstatistiksel Bulgular ve Sınıf Dağılımları

In [ ]:
# ===========================================================
# HÜCRE 9: F0 İstatistiksel Özet Tablosu
# ===========================================================
print('='*65)
print('  F0 İSTATİSTİKSEL ÖZET / F0 STATISTICAL SUMMARY')
print('='*65)

summary_rows = []
for cls in ['Erkek', 'Kadın', 'Çocuk']:
    subset = df_results[df_results['Cinsiyet'] == cls]
    f0_vals = subset['f0_mean'][subset['f0_mean'] > 50]
    
    if len(f0_vals) > 0:
        row = {
            'Sınıf'        : cls,
            'Örnek Sayısı' : len(subset),
            'Ort. F0 (Hz)' : f'{f0_vals.mean():.1f}',
            'Std F0 (Hz)'  : f'{f0_vals.std():.1f}',
            'Min F0 (Hz)'  : f'{f0_vals.min():.1f}',
            'Maks F0 (Hz)' : f'{f0_vals.max():.1f}',
            'Medyan F0'    : f'{f0_vals.median():.1f}',
        }
    else:
        row = {'Sınıf': cls, 'Örnek Sayısı': len(subset)}
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

In [ ]:
# ===========================================================
# HÜCRE 10: F0 Dağılım Grafikleri
# ===========================================================
fig = plot_f0_distributions(df_results)
plt.show()

---
## BÖLÜM 6: Kural Tabanlı Sınıflandırıcı ve Başarı Analizi

In [ ]:
# ===========================================================
# HÜCRE 11: Başarı Metrikleri
# ===========================================================
valid = df_results[
    df_results['Cinsiyet'].isin(['Erkek', 'Kadın', 'Çocuk']) &
    (df_results['prediction'] != 'Unknown')
]

metrics = compute_metrics(
    predictions=valid['prediction'].tolist(),
    ground_truth=valid['Cinsiyet'].tolist()
)

print(f'\n{"="*55}')
print(f'  GENEL DOĞRULUK / OVERALL ACCURACY: {metrics["overall_accuracy"]*100:.2f}%')
print(f'{"="*55}')
print(f'\n{"Sınıf":<10} {"N":>4} {"Recall":>8} {"Precision":>10} {"F1":>8}')
print('-'*44)
for cls in ['Erkek', 'Kadın', 'Çocuk']:
    pc = metrics['per_class'].get(cls, {})
    print(f'{cls:<10} {pc.get("support",0):>4} '
          f'{pc.get("recall",0)*100:>7.1f}% '
          f'{pc.get("precision",0)*100:>9.1f}% '
          f'{pc.get("f1_score",0):>8.3f}')

In [ ]:
# ===========================================================
# HÜCRE 12: Karışıklık Matrisi
# ===========================================================
fig = plot_confusion_matrix(
    cm=metrics['confusion_matrix'],
    class_labels=metrics['class_labels'],
    accuracy=metrics['overall_accuracy']
)
plt.show()

In [ ]:
# ===========================================================
# HÜCRE 13: Hata Analizi — Yanlış Tahminler
# ===========================================================
wrong = valid[~valid['correct']]
print(f'Yanlış tahmin sayısı: {len(wrong)}')

if len(wrong) > 0:
    print('\nYanlış tahminlerin detayları:')
    display_cols = ['Dosya_Path', 'Cinsiyet', 'prediction', 
                    'confidence', 'f0_mean', 'zcr_mean', 'reasoning']
    available = [c for c in display_cols if c in wrong.columns]
    print(wrong[available].to_string(index=False))
    
    print('\n📝 HATA ANALİZİ / ERROR ANALYSIS:')
    print('  Olası nedenler / Possible causes:')
    print('  1. F0 çakışma bölgesi (Kadın~Erkek: 155-180 Hz)')
    print('  2. Yüksek ses tonu (excited speech) — Erkek sesini Kadına kaydırır')
    print('  3. Büyük çocuk seslerinin Kadın aralığına girmesi')
    print('  4. Gürültü → Yanlış F0 tespiti')
    print('  5. Fısıltı / Unvoiced konuşma → F0 tespiti güçlüğü')

In [ ]:
# ===========================================================
# HÜCRE 14: Sonuçları CSV'ye Kaydet
# ===========================================================
os.makedirs('../results', exist_ok=True)
df_results.to_csv('../results/feature_results.csv', index=False)
print('✅ Sonuçlar kaydedildi: results/feature_results.csv')

# Özet tablo kaydet
df_summary.to_csv('../results/f0_summary_table.csv', index=False)
print('✅ F0 özet tablosu kaydedildi: results/f0_summary_table.csv')

---
## BÖLÜM 7: Otokorelasyon Araştırması — Matematiksel Analiz

### Otokorelasyon Fonksiyonu

$$R[\tau] = \sum_{n=0}^{N-\tau-1} x[n] \cdot x[n-\tau]$$

**Periyodik sinyaller için:** $\tau = T$ (temel periyot) değerinde maksimum değere ulaşır.  
**Temel frekans:** $F_0 = \frac{f_s}{\tau_{peak}}$

### FFT Tabanlı Hızlı Otokorelasyon

$$R[\tau] = \mathcal{F}^{-1}\{|X(f)|^2\}$$

Hesaplama karmaşıklığı: O(N log N) vs saf implementasyonda O(N²)

In [ ]:
# ===========================================================
# HÜCRE 15: Cinsiyet Sınıflarına Göre Otokorelasyon Karşılaştırması
# ===========================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Autocorrelation Functions by Gender Class\n'
             'Cinsiyet Sınıflarına Göre Otokorelasyon Fonksiyonları',
             fontsize=13, fontweight='bold')

class_colors = {'Erkek': '#2196F3', 'Kadın': '#E91E63', 'Çocuk': '#4CAF50'}

for ax, cls in zip(axes, ['Erkek', 'Kadın', 'Çocuk']):
    subset = df_results[(df_results['Cinsiyet'] == cls) & 
                        df_results['Dosya_Path'].apply(
                            lambda x: os.path.exists(str(x)))]
    
    if len(subset) == 0:
        ax.set_title(f'{cls} — No data')
        continue
    
    # İlk geçerli dosyanın en sesli çerçevesini al
    try:
        audio, sr_l = load_audio(str(subset.iloc[0]['Dosya_Path']), SR)
        frames_c, _, hl = frame_signal(audio, sr_l)
        ste_c = compute_short_time_energy(frames_c)
        zcr_c = compute_zcr(frames_c, sr_l)
        voiced_c = detect_voiced_frames(ste_c, zcr_c)
        
        if voiced_c.sum() > 0:
            vi = np.where(voiced_c)[0]
            bf = vi[np.argmax(ste_c[vi])]
            fr = frames_c[bf]
        else:
            fr = frames_c[np.argmax(ste_c)]
        
        acf_c = compute_autocorrelation_fast(fr)
        f0_c  = estimate_f0_autocorrelation(fr, sr_l)
        n = len(fr)
        lags = np.arange(n) / sr_l * 1000
        
        ax.plot(lags[:n//2], acf_c[:n//2], 
                color=class_colors[cls], linewidth=1.2)
        ax.set_title(f'{cls} Voice\nF0 ≈ {f0_c:.0f} Hz', fontsize=11)
        ax.set_xlabel('Lag (ms)')
        ax.set_ylabel('R(τ)')
        
        if f0_c > 0:
            period_ms = 1000/f0_c
            if period_ms < lags[n//2-1]:
                ax.axvline(period_ms, color='red', linestyle='--',
                           linewidth=1.5, label=f'T={period_ms:.1f}ms')
                ax.legend(fontsize=9)
    except Exception as e:
        ax.set_title(f'{cls} — Error: {e}')

plt.tight_layout()
plt.show()
print('📝 Gözlem: Her sınıfta ilk tepe noktasının gecikme değeri (T),'
      ' sınıfın F0 aralığını yansıtmaktadır.')